# SAC arrival_v2 — sac_asym s0_k4 SEED=0 replication of §7.7 (single_cross, 1M)

**Pre-context（commits `01b78ad` / `3d20e86`）**：[`docs/arrival_v2_experiment_report.md`](../docs/arrival_v2_experiment_report.md) §7.7 给出 AsymCritic pure ablation 的 negative finding（mean 0.044 vs vanilla 0.221，5× 更差），但**仅 seed=42 单 seed**。该 claim 在严格意义上是 single-point estimate，不是 well-established negative finding。

| 已闭环 single_cross cell | algo | sensor | k | seed | final | mean | OOB |
|---|---|---|---|---|---:|---:|---:|
| §7.1 single_cross_s1_k4 | vanilla | s1 | 4 | 42 | 0.900 | — | 0.100 |
| §7.6.4 single_cross_s0_k4 | vanilla | s0 | 4 | **42** | 0.100 | 0.221 | 0.667 |
| §7.7 single_cross_s0_k4 + asym | sac_asym | s0 | 4 | **42** | 0.167 | 0.044 | 0.633 |

**本 notebook 任务（sac_asym k=4 SEED=0 paired-seed replication）**：把 §7.7 的 sac_asym 配置**唯一变量**换成 `--seed 0`，其它**全部不动**：

| 维度 | §7.7 baseline (seed=42) | 本 notebook |
|---|---|---|
| `--seed` | 42 | **0** ← 唯一变量 |
| `--use-asymmetric-critic` | True | True |
| sensor layout | s0 (DVL-only) | s0 (DVL-only) |
| history_length | 4 | 4 |
| reward | arrival_v2 | arrival_v2 |
| flow U / target | 1.5 / 1.5 | 1.5 / 1.5 |
| total_steps | 1M | 1M |
| num_envs | 6 | 6 |
| benchmark | `single_u15_cross_tgt15` | 同 |
| obs_dim (actor) | 10×4 + 8 = 48 | 同 (48) |
| privileged_obs_dim (critic) | 2 | 同 (2) |

**配对设计**：本 notebook 与 `sac_arrival_v2_s0_cross_vanilla_seed0.ipynb` 并行跑，构成 2 seed × 2 algo = 4-cell paired ablation：

| | seed=42 (已有) | seed=0 (姊妹 + 本) |
|---|---|---|
| **vanilla** | §7.6.4 final=0.100 mean=0.221 | ← 姊妹 notebook |
| **sac_asym** | §7.7 final=0.167 mean=0.044 | ← **本 notebook** |

**判读规则**（驱动 §7.7 negative finding 升降级）：
- **HARDEN-NEGATIVE**（seed=0 asym mean ≤ 0.15 AND 显著低于 vanilla seed=0 / seed=42 均值）→ 2 seed × 2 algo 都验证 asym worse than vanilla → §7.7 升格为 **2-seed hardened negative claim**，可直接写进论文；§8 P1 路线维持「actor-side info upgrade」（k=8 / temporal）方向。
- **COLLAPSE-NEGATIVE**（seed=0 asym mean ∈ [0.15, 0.30]，与 vanilla baselines 同量级）→ §7.7 的「asym hurts」claim **collapse 为「asym does not improve」**，seed=42 的 mean 0.044 是 unlucky outlier；§8 P1 重写：asym 不再是"被推翻"，而是"被弱化为 no-effect"；下一步仍可走 k=8 但需补 3rd seed。
- **REVERSE**（seed=0 asym mean ≥ 0.30，明显高于 vanilla baselines）→ §7.7 claim **反转**：seed=42 是 asym 的 unlucky outlier，asym 可能确实有助益 → §7.7 整段重写，§8 P1 需要重新考虑 asym 作为候选。
- **AMBIGUOUS**（其他情况，例如 seed=0 asym mean 高于 §7.7 但仍低于 vanilla seeds）→ 部分确认；需要 3rd seed (=7 或 =1) 才能定论。

**输出根（与 §7.7 seed=42 互不覆盖）**：
- `experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_asym/s0_k4/seed_0/`

**总预算**：~2.5h L4（1 Colab Pro+ session）。可与姊妹 vanilla seed=0 + 已在跑的 k=8 共 3 session 并行。

**风格**：训练用 `!python -u -m scripts.train_sac` 直跑（实时 stdout），与 §7 / §7.6 / §7.7 一致。


## 0. GPU sanity


In [ ]:
!nvidia-smi | head -10


## 1. Mount Drive + cwd


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

REPO_DIR = '/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5'
%cd $REPO_DIR


## 2. Config — sac_asym + seed=0（与 §7.7 baseline 仅差一个 flag value）


In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

# ==== SAC / env config (与 §7.7 baseline 严格一致，除 seed 外) ====
OBJECTIVE = 'arrival_v2'
PROBE_LAYOUT = 's0'
HISTORY_LENGTH = 4
TARGET_SPEED = 1.5
SEED = 0                                # ← 唯一与 §7.7 不同 (was 42)

RANDOM_STEPS = 5_000
UPDATE_AFTER = 5_000
BATCH_SIZE = 256
HIDDEN_DIM = 256
NUM_ENVS = 6
EVAL_EVERY = 25_000
EVAL_EPISODES = 30
CHECKPOINT_EVERY = 100_000
DEVICE = 'cuda'

# AsymCritic 关键 flag — 与 §7.7 一致，pure ablation（不带 LN / UTD / Dropout）
USE_ASYMMETRIC_CRITIC = True           # ← 与 vanilla 路线唯一差别
USE_LAYERNORM = False
UPDATES_PER_STEP = 1
DROPOUT_RATE = 0.0

PASS_FINAL_SUCCESS = 0.85
PASS_LAST100_RATIO = 0.90
PASS_OOB_RATE = 0.10

# Flow file（与 §7.1 / §7.6.4 / §7.7 严格一致）
SINGLE_FLOW = 'wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy'

# ==== Single phase — sac_asym s0 k=4 seed=0 (X_*) ====
X_BENCHMARK_KEY = 'single_u15_cross_tgt15'
X_TASK_GEOMETRY = 'cross_stream'
X_FLOW_PATH = SINGLE_FLOW
X_TOTAL_STEPS = 1_000_000
X_RUN_ROOT = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_asym/s0_k4/seed_0')
X_CKPT_ROOT = Path('checkpoints/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_asym/s0_k4/seed_0')
X_MANIFEST_PATH = Path(f'benchmarks/{X_BENCHMARK_KEY}.json')

# Baselines（事后对比）
X_K4_ASYM_SEED42_ANCHOR = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_asym/s0_k4/seed_42')   # §7.7
X_K4_V_SEED42_BASELINE  = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_42') # §7.6.4
X_K4_V_SEED0_SISTER     = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_0')  # 姊妹 notebook
X_S1_BASELINE           = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42') # §7.1 上界 ref

os.environ['PYTHONUNBUFFERED'] = '1'

print(f'PROBE_LAYOUT          = {PROBE_LAYOUT}')
print(f'OBJECTIVE             = {OBJECTIVE}')
print(f'HISTORY_LENGTH        = {HISTORY_LENGTH}')
print(f'SEED                  = {SEED}        ← 唯一与 §7.7 不同 (was 42)')
print(f'NUM_ENVS              = {NUM_ENVS}')
print(f'USE_ASYMMETRIC_CRITIC = {USE_ASYMMETRIC_CRITIC}   ← 与 §7.7 一致')
print(f'USE_LAYERNORM         = {USE_LAYERNORM}')
print(f'UPDATES_PER_STEP      = {UPDATES_PER_STEP}')
print(f'DROPOUT_RATE          = {DROPOUT_RATE}')
print()
print(f'benchmark             : {X_BENCHMARK_KEY}')
print(f'geometry              : {X_TASK_GEOMETRY}')
print(f'total_steps           : {X_TOTAL_STEPS:,}')
print(f'expected obs_dim      : 10 * {HISTORY_LENGTH} + 8 (arrival_v2 context) = {10 * HISTORY_LENGTH + 8}')
print(f'expected privileged   : dim=2 (body-frame [u_eq, v_eq])')
print(f'run_root              : {X_RUN_ROOT}')
print(f'§7.7 seed=42 anchor   : {X_K4_ASYM_SEED42_ANCHOR}')
print(f'§7.6.4 vanilla ref    : {X_K4_V_SEED42_BASELINE}')
print(f'sister vanilla seed=0 : {X_K4_V_SEED0_SISTER}')
print(f'§7.1 s1_k4 upper ref  : {X_S1_BASELINE}')


## 3. Preflight — flow / arrival_v2 candidate gate / reward unit tests / manifest / baseline 就位


In [ ]:
# Flow file
fp = Path(X_FLOW_PATH)
if not fp.exists():
    raise FileNotFoundError(f'missing flow file: {fp}')
print(f'[OK] flow file: {fp}  ({fp.stat().st_size / 1e6:.1f} MB)')


In [ ]:
!python -u -m scripts.validate_arrival_v2_candidate


In [ ]:
!python -u -m pytest tests/test_reward_objective.py -q


In [ ]:
if not X_MANIFEST_PATH.exists():
    !python -u -m scripts.generate_standard_benchmarks --benchmarks {X_BENCHMARK_KEY} --episodes {EVAL_EPISODES}
if not X_MANIFEST_PATH.exists():
    raise FileNotFoundError(f'manifest not generated: {X_MANIFEST_PATH}')
print(f'[OK] manifest ready: {X_MANIFEST_PATH}')


In [ ]:
# 检查对比 baseline 是否就位 — 不影响训练，仅 §6 verdict 用
for label, root, ref_final, ref_oob in [
    ('§7.7   sac_asym_s0_k4 seed=42 (paired) ', X_K4_ASYM_SEED42_ANCHOR, 0.167, 0.633),
    ('§7.6.4 vanilla_s0_k4 seed=42           ', X_K4_V_SEED42_BASELINE,  0.100, 0.667),
    ('§7.1   vanilla_s1_k4 seed=42 (upper)   ', X_S1_BASELINE,           0.900, 0.100),
    ('SISTER vanilla_s0_k4 seed=0            ', X_K4_V_SEED0_SISTER,     -1,    -1),
]:
    fp = root / 'results' / 'final_eval.json'
    if fp.exists():
        d = json.loads(fp.read_text(encoding='utf-8'))
        f = float(d['eval_success_rate'])
        c = d.get('eval_termination_counts', {})
        n = float(d.get('num_eval_episodes', EVAL_EPISODES))
        oob = float(c.get('out_of_bounds', 0)) / max(n, 1.0)
        if ref_final >= 0:
            match = '✓' if abs(f - ref_final) < 0.05 and abs(oob - ref_oob) < 0.05 else '✗ mismatch'
        else:
            match = '(sister, no ref)'
        print(f'[OK] {label}: final={f:.4f}  oob={oob:.4f}  counts={c}  {match}')
    else:
        if 'SISTER' in label:
            print(f'[INFO] {label}: 姊妹 notebook 尚未跑完，§6 verdict 会跳过 2x2 完整设计')
        else:
            print(f'[WARN] {label}: {fp} 不存在 (后续 §6 会回退到 report 转载值)')


## 4. Train — sac_asym s0 k=4 SEED=0 (1.0M, skip/resume)


In [ ]:
x_state_path = X_RUN_ROOT / 'trainer_state.json'
if x_state_path.exists():
    x_state = json.loads(x_state_path.read_text(encoding='utf-8'))
    x_current_step = int(x_state.get('env_step', 0))
else:
    x_current_step = 0
print(f'[state] X env_step = {x_current_step:,} / target {X_TOTAL_STEPS:,}')

if x_current_step >= X_TOTAL_STEPS:
    print(f'[skip] X already trained to {x_current_step:,} >= {X_TOTAL_STEPS:,}')
elif x_current_step > 0:
    print(f'[resume] X continuing from {x_current_step:,} -> {X_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --resume {str(X_RUN_ROOT)} \
        --total-steps {X_TOTAL_STEPS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {str(X_MANIFEST_PATH)} \
        --device {DEVICE}
else:
    print(f'[train] X fresh start -> {X_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --flow {X_FLOW_PATH} \
        --task-geometry {X_TASK_GEOMETRY} \
        --target-speed {TARGET_SPEED} \
        --objective {OBJECTIVE} \
        --probe-layout {PROBE_LAYOUT} \
        --history-length {HISTORY_LENGTH} \
        --use-asymmetric-critic \
        --total-steps {X_TOTAL_STEPS} \
        --random-steps {RANDOM_STEPS} \
        --update-after {UPDATE_AFTER} \
        --batch-size {BATCH_SIZE} \
        --hidden-dim {HIDDEN_DIM} \
        --num-envs {NUM_ENVS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {str(X_MANIFEST_PATH)} \
        --seed {SEED} \
        --device {DEVICE} \
        --save-dir {str(X_RUN_ROOT)} \
        --checkpoint-dir {str(X_CKPT_ROOT)}


## 5. Summary + gate


In [ ]:
def summarize_phase(run_root: Path, total_steps: int, label: str, gate_filename: str):
    eval_log_path = run_root / 'results' / 'eval_log.csv'
    final_eval_path = run_root / 'results' / 'final_eval.json'
    trainer_state_path = run_root / 'trainer_state.json'
    train_config_path = run_root / 'results' / 'train_config.txt'

    if not eval_log_path.exists():
        raise FileNotFoundError(f'missing eval log: {eval_log_path}')
    if not final_eval_path.exists():
        raise FileNotFoundError(f'missing final eval: {final_eval_path}')

    df = pd.read_csv(eval_log_path)
    final_eval = json.loads(final_eval_path.read_text(encoding='utf-8'))
    trainer_state = json.loads(trainer_state_path.read_text(encoding='utf-8')) if trainer_state_path.exists() else {}

    # Parse train_config.txt for seed / history_length / privileged_obs_dim confirmation
    # (trainer_state.json 不 dump use_asymmetric_critic key — 必须从 train_config.txt 读 privileged_obs_dim=2 来确认)
    seed_from_config = 'NA'
    history_from_config = 'NA'
    privileged_dim_from_config = 'NA'
    if train_config_path.exists():
        for ln in train_config_path.read_text(encoding='utf-8').splitlines():
            s = ln.strip()
            if s.startswith('seed='):
                seed_from_config = s.split('=', 1)[1]
            elif s.startswith('history_length='):
                history_from_config = s.split('=', 1)[1]
            elif s.startswith('privileged_obs_dim='):
                privileged_dim_from_config = s.split('=', 1)[1]

    peak_success = float(df['eval_success_rate'].max()) if len(df) else 0.0
    peak_step = int(df.loc[df['eval_success_rate'].idxmax(), 'env_step']) if len(df) else 0
    last100 = df[df['env_step'] >= total_steps - 100_000].copy()
    last100_mean = float(last100['eval_success_rate'].mean()) if len(last100) else 0.0
    mean_all = float(df['eval_success_rate'].mean()) if len(df) else 0.0
    n_evals_with_success = int((df['eval_success_rate'] > 0).sum()) if len(df) else 0
    final_success = float(final_eval['eval_success_rate'])
    counts = final_eval.get('eval_termination_counts', {})
    num_eps = float(final_eval.get('num_eval_episodes', EVAL_EPISODES))
    oob_rate = float(counts.get('out_of_bounds', 0)) / max(num_eps, 1.0)

    print('=' * 100)
    print(f'{label}  (arrival_v2 / s0 / k=4 / sac_asym / seed=0 / {total_steps:,} steps)')
    print('-' * 100)
    print(f"  final_success_rate    : {final_success:.4f}   gate >= {PASS_FINAL_SUCCESS:.2f}")
    print(f"  peak_success_rate     : {peak_success:.4f}   @ {peak_step:,}")
    print(f"  last100k_mean_success : {last100_mean:.4f}   gate >= {PASS_LAST100_RATIO * peak_success:.4f}")
    print(f"  full-traj mean        : {mean_all:.4f}   ({len(df)} evals)")
    print(f"  n_evals_with_success  : {n_evals_with_success} / {len(df)}")
    print(f"  final_oob_rate        : {oob_rate:.4f}   gate <= {PASS_OOB_RATE:.2f}")
    print(f"  obs_dim               : {trainer_state.get('observation_dim', 'NA')}   (expect 48)")
    print(f"  seed                  : {seed_from_config}   (expect 0)")
    print(f"  history_length        : {history_from_config}   (expect 4)")
    print(f"  privileged_obs_dim    : {privileged_dim_from_config}   (expect 2 — confirms asym critic wired)")
    print(f"  context_obs           : {trainer_state.get('include_episode_context_obs', 'NA')}")
    print(f"  timeout_bootstrap     : {trainer_state.get('timeout_bootstrap_semantics', 'NA')}")
    print(f"  termination           : {counts}")
    print('=' * 100)

    if str(privileged_dim_from_config) != '2':
        print('⚠️  privileged_obs_dim ≠ 2 — AsymCritic 没生效！请检查 CLI flag。')

    if len(df):
        print()
        print('[last 16 eval rows]')
        cols = ['env_step', 'eval_success_rate', 'eval_return', 'eval_safety_cost',
                'eval_time_s', 'eval_progress_ratio']
        available = [c for c in cols if c in df.columns]
        print(df[available].tail(16).to_string(index=False))

    checks = [
        ('final success >= 0.85', final_success >= PASS_FINAL_SUCCESS, f'{final_success:.4f}'),
        ('last100k mean >= 0.9 * peak', last100_mean >= PASS_LAST100_RATIO * peak_success,
         f'{last100_mean:.4f} / peak={peak_success:.4f}'),
        ('final OOB rate <= 0.10', oob_rate <= PASS_OOB_RATE, f'{oob_rate:.4f}'),
        ('arrival_v2 context obs enabled',
         trainer_state.get('include_episode_context_obs') is True,
         str(trainer_state.get('include_episode_context_obs'))),
        ('arrival_v2 timeout terminal semantics',
         trainer_state.get('timeout_bootstrap_semantics') == 'terminal',
         str(trainer_state.get('timeout_bootstrap_semantics'))),
        ('asym critic wired (privileged_obs_dim=2)',
         str(privileged_dim_from_config) == '2',
         f'privileged_obs_dim={privileged_dim_from_config}'),
    ]

    print()
    print('=' * 96)
    print(f"{'check':<48}{'pass':>8}{'detail':>40}")
    print('-' * 96)
    all_pass = True
    for name, ok, detail in checks:
        mark = 'PASS' if ok else 'FAIL'
        if not ok:
            all_pass = False
        print(f'{name:<48}{mark:>8}{detail:>40}')
    print('=' * 96)

    summary = {
        'phase': label,
        'objective': OBJECTIVE,
        'probe_layout': PROBE_LAYOUT,
        'history_length': int(history_from_config) if history_from_config != 'NA' else HISTORY_LENGTH,
        'seed': int(seed_from_config) if seed_from_config != 'NA' else SEED,
        'total_steps': total_steps,
        'algorithm': 'sac_asym',
        'privileged_obs_dim': int(privileged_dim_from_config) if str(privileged_dim_from_config).isdigit() else None,
        'final_success_rate': final_success,
        'peak_success_rate': peak_success,
        'peak_step': peak_step,
        'last100k_mean_success': last100_mean,
        'mean_success_full_trajectory': mean_all,
        'n_evals_with_success': n_evals_with_success,
        'n_evals_total': len(df),
        'final_oob_rate': oob_rate,
        'termination_counts': counts,
        'all_pass': bool(all_pass),
        'checks': [{'name': n, 'ok': bool(ok), 'detail': d} for n, ok, d in checks],
    }
    out_path = run_root / 'results' / gate_filename
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(f'[saved] {out_path}')
    return summary

x_summary = summarize_phase(
    X_RUN_ROOT,
    X_TOTAL_STEPS,
    'SINGLE_CROSS_SAC_ASYM_S0_K4_SEED0',
    'sac_asym_s0_k4_seed0_gate_summary.json',
)
X_PASS = x_summary['all_pass']
print()
print(f'X_PASS = {X_PASS}')


## 6. Seed-pair verdict — asym seed=0 vs seed=42（验证 §7.7 negative finding 是否 seed-stable）


In [ ]:
def read_baseline(root: Path, ref_final: float, ref_oob: float, ref_mean: float = None):
    fp = root / 'results' / 'final_eval.json'
    log = root / 'results' / 'eval_log.csv'
    out = {'final': ref_final, 'oob': ref_oob, 'mean': ref_mean, 'peak': None, 'source': 'report-fallback'}
    if fp.exists():
        d = json.loads(fp.read_text(encoding='utf-8'))
        f = float(d['eval_success_rate'])
        c = d.get('eval_termination_counts', {})
        n = float(d.get('num_eval_episodes', EVAL_EPISODES))
        out['final'] = f
        out['oob'] = float(c.get('out_of_bounds', 0)) / max(n, 1.0)
        out['source'] = 'on-disk'
    if log.exists():
        dlog = pd.read_csv(log)
        out['mean'] = float(dlog['eval_success_rate'].mean())
        out['peak'] = float(dlog['eval_success_rate'].max())
    return out

print('=' * 100)
print('SAC_ASYM s0_k4 — paired seed verdict (seed=0 vs seed=42)')
print('-' * 100)

# 当前 seed=0 结果
a0_final = float(x_summary['final_success_rate'])
a0_oob = float(x_summary['final_oob_rate'])
a0_peak = float(x_summary['peak_success_rate'])
a0_mean = float(x_summary.get('mean_success_full_trajectory', 0.0))
a0_n_succ = int(x_summary.get('n_evals_with_success', 0))
a0_n_tot = int(x_summary.get('n_evals_total', 0))

# Baselines
b_a42 = read_baseline(X_K4_ASYM_SEED42_ANCHOR, 0.167, 0.633, 0.044)
b_v42 = read_baseline(X_K4_V_SEED42_BASELINE,  0.100, 0.667, 0.221)
b_v0  = read_baseline(X_K4_V_SEED0_SISTER,     -1,    -1,    None)  # 姊妹 notebook
b_s1  = read_baseline(X_S1_BASELINE,           0.900, 0.100, None)

# 计算 2-seed × 2-algo cell mean
asym_2seed_mean = (a0_mean + (b_a42['mean'] or 0.044)) / 2.0
if b_v0['source'] == 'on-disk':
    vanilla_2seed_mean = (b_v0['mean'] + (b_v42['mean'] or 0.221)) / 2.0
else:
    vanilla_2seed_mean = None

print()
print(f'{"config":<48}{"final":>10}{"mean":>10}{"oob":>10}{"source":>14}')
print('-' * 100)
print(f'{"vanilla SAC, s1_k4 seed=42 (§7.1 upper ref)":<48}{b_s1["final"]:>10.4f}{(b_s1["mean"] or float("nan")):>10.4f}{b_s1["oob"]:>10.4f}{b_s1["source"]:>14}')
print(f'{"vanilla SAC, s0_k4 seed=42 (§7.6.4)":<48}{b_v42["final"]:>10.4f}{(b_v42["mean"] or float("nan")):>10.4f}{b_v42["oob"]:>10.4f}{b_v42["source"]:>14}')
if b_v0['source'] == 'on-disk':
    print(f'{"vanilla SAC, s0_k4 SEED=0 (sister run)":<48}{b_v0["final"]:>10.4f}{(b_v0["mean"] or float("nan")):>10.4f}{b_v0["oob"]:>10.4f}{b_v0["source"]:>14}')
else:
    print(f'{"vanilla SAC, s0_k4 SEED=0 (sister run)":<48}{"PENDING":>10}{"PENDING":>10}{"PENDING":>10}{"not-yet":>14}')
print(f'{"sac_asym,   s0_k4 seed=42 (§7.7 anchor)":<48}{b_a42["final"]:>10.4f}{(b_a42["mean"] or float("nan")):>10.4f}{b_a42["oob"]:>10.4f}{b_a42["source"]:>14}')
print(f'{"sac_asym,   s0_k4 SEED=0 (this run)":<48}{a0_final:>10.4f}{a0_mean:>10.4f}{a0_oob:>10.4f}{"on-disk":>14}')
print('=' * 100)

# Δ 行
print()
print(f'{"contrast":<60}{"Δ final":>12}{"Δ mean":>12}{"Δ oob":>12}')
print('-' * 100)
d_final = a0_final - b_a42['final']
d_mean = a0_mean - (b_a42['mean'] or 0)
d_oob = a0_oob - b_a42['oob']
print(f'{"asym seed=0 vs seed=42 — paired test":<60}{d_final:>+12.4f}{d_mean:>+12.4f}{d_oob:>+12.4f}')
print(f'{"asym seed=0 vs vanilla seed=42 — cross-arm":<60}'
      f'{a0_final - b_v42["final"]:>+12.4f}'
      f'{a0_mean - (b_v42["mean"] or 0):>+12.4f}'
      f'{a0_oob - b_v42["oob"]:>+12.4f}')
if b_v0['source'] == 'on-disk':
    print(f'{"asym seed=0 vs vanilla seed=0 — same-seed cross-arm":<60}'
          f'{a0_final - b_v0["final"]:>+12.4f}'
          f'{a0_mean - (b_v0["mean"] or 0):>+12.4f}'
          f'{a0_oob - b_v0["oob"]:>+12.4f}')
    print(f'{"2-seed asym mean vs 2-seed vanilla mean":<60}'
          f'{"N/A":>12}'
          f'{asym_2seed_mean - vanilla_2seed_mean:>+12.4f}'
          f'{"N/A":>12}')
print('=' * 100)
print()
print(f'seed=0 evals_with_success: {a0_n_succ} / {a0_n_tot}  (seed=42 asym was 19/39, vanilla was 35/39)')
print(f'2-seed asym mean : {asym_2seed_mean:.4f}')
if vanilla_2seed_mean is not None:
    print(f'2-seed vanilla mean: {vanilla_2seed_mean:.4f}')
    print(f'asym - vanilla (2-seed mean delta): {asym_2seed_mean - vanilla_2seed_mean:+.4f}')

# 判读 — 4 档 paired-seed verdict for §7.7
vanilla_ref_mean = vanilla_2seed_mean if vanilla_2seed_mean is not None else (b_v42['mean'] or 0.221)

if a0_mean <= 0.15 and a0_mean < vanilla_ref_mean - 0.05:
    verdict = ('HARDEN-NEGATIVE — seed=0 asym mean={:.3f} 仍显著低于 vanilla 参照 ({:.3f})；'
               '配合 §7.7 的 seed=42 mean=0.044，2 seed × 2 algo 都证实 asym worse than vanilla；'
               '§7.7 升格为 2-seed hardened negative claim ✓；§8 P1 路线维持 actor-side info upgrade (k=8 / temporal)'
               .format(a0_mean, vanilla_ref_mean))
elif a0_mean >= 0.30 and a0_mean > vanilla_ref_mean + 0.05:
    verdict = ('REVERSE — seed=0 asym mean={:.3f} 明显高于 vanilla 参照 ({:.3f})；'
               '§7.7 claim 反转：seed=42 是 asym 的 unlucky outlier，asym 可能确实有助益；'
               '§7.7 整段重写，§8 P1 需重新考虑 asym 作为候选'.format(a0_mean, vanilla_ref_mean))
elif abs(a0_mean - vanilla_ref_mean) <= 0.10:
    verdict = ('COLLAPSE-NEGATIVE — seed=0 asym mean={:.3f} 与 vanilla 参照 ({:.3f}) 同量级；'
               '§7.7 的「asym hurts」claim collapse 为「asym does not improve」；'
               'seed=42 mean 0.044 是 unlucky outlier；§8 P1 重写 asym 段为 no-effect；下一步仍可走 k=8 但需补 3rd seed'
               .format(a0_mean, vanilla_ref_mean))
else:
    verdict = ('AMBIGUOUS — seed=0 asym mean={:.3f}，处于 §7.7 anchor (0.044) 与 vanilla ref ({:.3f}) 之间；'
               '部分确认 §7.7 trend；需要 3rd seed (=7 或 =1) 才能定论'
               .format(a0_mean, vanilla_ref_mean))

print()
print(f'>>> verdict: {verdict}')

# 落盘
ablation_out = {
    'experiment': 'arrival_v2_s0_cross_sac_asym_seed0_replication',
    'seed': SEED,
    'benchmark': X_BENCHMARK_KEY,
    'probe_layout': PROBE_LAYOUT,
    'history_length': HISTORY_LENGTH,
    'total_steps': X_TOTAL_STEPS,
    'algorithm': 'sac_asym',
    'cli_diff_vs_§7.7_baseline': '--seed 42 → --seed 0',
    'results': {
        'sac_asym_s0_k4_seed0_thisrun': {
            'final': a0_final, 'mean': a0_mean, 'oob': a0_oob, 'peak': a0_peak,
            'n_evals_with_success': a0_n_succ, 'n_evals_total': a0_n_tot,
        },
        'sac_asym_s0_k4_seed42_anchor_§7.7': b_a42,
        'vanilla_s0_k4_seed42_§7.6.4': b_v42,
        'vanilla_s0_k4_seed0_sister': b_v0,
        'vanilla_s1_k4_seed42_§7.1_upper': b_s1,
    },
    'delta_vs_§7.7_seed42_paired': {
        'final_pp': round(d_final * 100, 2),
        'mean_pp': round(d_mean * 100, 2),
        'oob_pp': round(d_oob * 100, 2),
    },
    'two_seed_summary': {
        'asym_2seed_mean': asym_2seed_mean,
        'vanilla_2seed_mean': vanilla_2seed_mean,
        'asym_minus_vanilla_2seed_mean': (
            asym_2seed_mean - vanilla_2seed_mean if vanilla_2seed_mean is not None else None
        ),
    },
    'all_pass': bool(x_summary['all_pass']),
    'verdict': verdict,
}
out_dir = Path('experiments/arrival_v2_prototype/s0_cross_sac_asym_seed0_summary')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'combined_gate_summary.json'
out_path.write_text(json.dumps(ablation_out, indent=2), encoding='utf-8')
print(f'\n[saved] {out_path}')
